[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/B.%20Core%20Quay-Side%20Operations%20%28The%20Ship-to-Shore%20Interface%29/10.%20The%20Dual-Cycling%20Quay%20Crane%20Problem/P10-Tier-4_executed.ipynb)

---


# 10. The Dual-Cycling Quay Crane Problem

## Tier 4 — Imitation Learning for Crane Operations

### Goal

This notebook demonstrates behavioral cloning: a supervised-learning approach that trains a policy to imitate decisions made by a teacher. The example uses synthetic teacher demonstrations to illustrate how state features, action labels, training, and evaluation fit together. It does not use operational records or an optimal solver.

### Inputs and outputs

The input to the policy is a fixed-length vector derived from a crane-operation state. The action representation contains a four-class operation choice and four continuous or binary parameters. Training uses the teacher's operation class and those four parameters. The notebook reports held-out action accuracy, a dual-cycle-flag prediction score, and measured inference time.

### Scope and modeling assumptions

- A synthetic teacher rule supplies consistent labels; these labels are not evidence of real expert performance or optimal decisions.
- The generated states are independent samples, not transitions from a vessel simulator. The notebook therefore cannot evaluate schedule quality, makespan, safety, or operational feasibility.
- Behavioral cloning is implemented and trained with gradient descent. Inverse reinforcement learning (IRL) and Dataset Aggregation (DAgger) are described as extensions, not reported as implemented results: IRL requires a defined trajectory-comparison training signal, while DAgger requires an interactive environment and an expert oracle for queried states.
- The learned action is a prediction, not a validated crane command. Operational constraints and safety checks are outside this model.

### Learning sequence

1. Generate synthetic state-action demonstrations from a transparent teacher rule.
2. Convert states and actions into fixed-size numeric representations.
3. Train a neural network by behavioral cloning.
4. Evaluate action predictions on trajectories held out from training.
5. Compare the policy with the teacher rule and inspect diagnostic plots.

The test split is made by trajectory, so states from one generated trajectory do not appear in both the training and test sets. Since each trajectory is synthetic and independently generated, this is a demonstration of evaluation procedure, not a claim of generalization to new terminals.

In [ ]:
import random
import time
from dataclasses import dataclass
from typing import Dict, List, Tuple

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# These seeds make the synthetic sample and network initialization reproducible.
np.random.seed(42)
random.seed(42)

plt.style.use("seaborn-v0_8")
sns.set_palette("husl")

## Mathematical foundation

### Behavioral cloning objective

At each decision step, the teacher supplies an operation label and four action parameters. The network predicts both: a probability distribution over the four operation types and a real-valued vector for the remaining encoded action components. The training objective combines categorical cross-entropy with a mean-squared error term:

$$
L_{BC}(\theta) = -\frac{1}{B}\sum_{i=1}^{B}\sum_{k=0}^{3} y_{ik}\log p_{ik}
+ \frac{\lambda}{2BD}\sum_{i=1}^{B}\sum_{j=1}^{D}(\hat{u}_{ij}-u_{ij})^2.
$$

Here, $B$ is the minibatch size; $y_{ik}$ is the one-hot teacher label for operation type $k$; $p_{ik}$ is the predicted probability for that type; $u_{ij}$ and $\hat{u}_{ij}$ are the teacher and predicted continuous action components; $D=4$; and $\lambda=0.5$ is the weight used in the code. The parameter vector $\theta$ contains all network weights and biases. Minimizing this loss encourages the predicted operation to match the teacher and the other action components to be numerically close to their labels.

### State and action representations

A state is mapped to a vector $x_t\in\mathbb{R}^{42}$. Its entries encode crane positions, load indicators and recent operation times; vessel workload, stability and completion; hatch states; and operational and environmental factors. Features that are not present in a particular state representation are padded with zeros. Standardization parameters are fitted using training data only, then reused for test data and predictions.

The encoded action is

$$
a_t = (c_t, o_t, q_t, r_t, \tau_t, d_t, s_t, i_t),
$$

where $c_t$ is the crane identifier, $o_t\in\{0,1,2,3\}$ is the operation type (unload, load, move, or wait), $q_t$ is the target-container identifier, $r_t$ is priority, $\tau_t$ is estimated time, $d_t$ indicates whether the teacher flags a dual cycle, $s_t$ is stability impact, and $i_t$ is interference risk. The classifier learns $o_t$; the regression head learns $(\tau_t,d_t,s_t,i_t)$. The other encoded fields are retained in the action representation but are not training targets in this implementation.

### Training and inference

A neural network transforms each standardized state through two ReLU hidden layers. Its output has eight values: four logits for operation type and four regression predictions. Softmax converts the logits into class probabilities. For each minibatch, backpropagation computes the gradient of $L_{BC}$ and gradient descent updates the network parameters. At inference time, the operation prediction is the class with the greatest probability; the regression head returns the four predicted parameters.

### IRL and DAgger context

Inverse reinforcement learning seeks a reward function that explains expert behavior, often by preferring expert trajectories $\tau^E$ to alternatives $\tau^R$. A representative margin objective is $\max(0, m - [R(\tau^E)-R(\tau^R)])$, where $m>0$ is a margin and $R(\tau)$ is trajectory reward. DAgger iteratively adds expert-labeled states encountered by the learned policy, addressing the distribution shift that can arise when a policy visits states absent from its demonstrations. Neither procedure is trained here: the synthetic data generator provides neither a transition simulator nor an independent optimal expert oracle. Accordingly, the numerical results below concern behavioral cloning only.

In [ ]:
@dataclass
class ExpertState:
    """State variables used to form a synthetic crane-operation example."""

    crane_positions: List[Tuple[float, float]]
    crane_loads: List[bool]
    crane_last_operations: List[float]
    vessel_import_containers: int
    vessel_export_containers: int
    vessel_stability_margin: float
    vessel_completion: float
    hatch_states: List[bool]
    truck_queue_length: int
    yard_utilization: float
    weather_factor: float
    time_pressure: float
    current_dual_ratio: float
    estimated_makespan: float
    priority_containers: int
    interference_risk: float


@dataclass
class ExpertAction:
    """Teacher action, including an operation class and associated attributes."""

    crane_id: int
    operation_type: int
    target_container: int
    priority_level: float
    estimated_time: float
    enables_dual_cycle: bool
    stability_impact: float
    interference_risk: float


@dataclass
class ExpertTrajectory:
    """A sequence of synthetic states, teacher actions, and illustrative rewards."""

    states: List[ExpertState]
    actions: List[ExpertAction]
    rewards: List[float]
    trajectory_id: int


class SimpleNeuralNetwork:
    """Small NumPy network trained with minibatch gradient descent."""

    def __init__(self, input_dim: int, hidden_dim: int, output_dim: int):
        if input_dim <= 0 or hidden_dim < 2 or output_dim != 8:
            raise ValueError("Expected positive input size, hidden_dim >= 2, and output_dim = 8.")

        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.output_dim = output_dim
        second_hidden_dim = hidden_dim // 2

        self.W1 = np.random.randn(input_dim, hidden_dim) * 0.1
        self.b1 = np.zeros(hidden_dim)
        self.W2 = np.random.randn(hidden_dim, second_hidden_dim) * 0.1
        self.b2 = np.zeros(second_hidden_dim)
        self.W3 = np.random.randn(second_hidden_dim, output_dim) * 0.1
        self.b3 = np.zeros(output_dim)
        self.lr = 0.001

    @staticmethod
    def relu(values: np.ndarray) -> np.ndarray:
        return np.maximum(0.0, values)

    @staticmethod
    def softmax(logits: np.ndarray) -> np.ndarray:
        shifted = logits - np.max(logits, axis=1, keepdims=True)
        exp_values = np.exp(shifted)
        return exp_values / np.sum(exp_values, axis=1, keepdims=True)

    def forward(self, inputs: np.ndarray) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
        z1 = inputs @ self.W1 + self.b1
        h1 = self.relu(z1)
        z2 = h1 @ self.W2 + self.b2
        h2 = self.relu(z2)
        output = h2 @ self.W3 + self.b3
        return output, z1, h1, z2, h2

    def predict(self, inputs: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
        output, _, _, _, _ = self.forward(inputs)
        probabilities = self.softmax(output[:, :4])
        continuous_parameters = output[:, 4:]
        return probabilities, continuous_parameters

    def backward(
        self,
        inputs: np.ndarray,
        target_discrete: np.ndarray,
        target_continuous: np.ndarray,
    ) -> float:
        """Compute the stated loss gradient and apply one SGD update."""
        batch_size = inputs.shape[0]
        if batch_size == 0:
            raise ValueError("A training batch must contain at least one example.")

        output, z1, h1, z2, h2 = self.forward(inputs)
        probabilities = self.softmax(output[:, :4])
        continuous_predictions = output[:, 4:]

        epsilon = 1e-8
        discrete_loss = -np.mean(
            np.sum(target_discrete * np.log(probabilities + epsilon), axis=1)
        )
        continuous_loss = np.mean((continuous_predictions - target_continuous) ** 2)
        loss = discrete_loss + 0.5 * continuous_loss

        # The softmax-cross-entropy derivative and MSE derivative provide the
        # learning signal that updates parameters toward teacher actions.
        output_gradient = np.zeros_like(output)
        output_gradient[:, :4] = (probabilities - target_discrete) / batch_size
        output_gradient[:, 4:] = (
            continuous_predictions - target_continuous
        ) / (batch_size * target_continuous.shape[1])

        dW3 = h2.T @ output_gradient
        db3 = np.sum(output_gradient, axis=0)
        dh2 = output_gradient @ self.W3.T
        dz2 = dh2 * (z2 > 0)
        dW2 = h1.T @ dz2
        db2 = np.sum(dz2, axis=0)
        dh1 = dz2 @ self.W2.T
        dz1 = dh1 * (z1 > 0)
        dW1 = inputs.T @ dz1
        db1 = np.sum(dz1, axis=0)

        self.W1 -= self.lr * dW1
        self.b1 -= self.lr * db1
        self.W2 -= self.lr * dW2
        self.b2 -= self.lr * db2
        self.W3 -= self.lr * dW3
        self.b3 -= self.lr * db3
        return float(loss)


In [ ]:
class DualCycleImitationLearner:
    """Behavioral-cloning learner for the synthetic dual-cycling example."""

    def __init__(self, state_dim: int = 42, action_dim: int = 8, hidden_dim: int = 256):
        if state_dim <= 0 or action_dim != 8:
            raise ValueError("state_dim must be positive and action_dim must equal 8.")

        self.state_dim = state_dim
        self.action_dim = action_dim
        self.hidden_dim = hidden_dim
        self.bc_network = SimpleNeuralNetwork(state_dim, hidden_dim, action_dim)
        self.state_scaler = StandardScaler()
        self.is_trained = False
        self.training_history = {"bc_loss": [], "accuracy": []}

    def extract_state_features(self, raw_state: ExpertState) -> np.ndarray:
        """Map state fields into a fixed-length numeric vector."""
        if not (
            len(raw_state.crane_positions)
            == len(raw_state.crane_loads)
            == len(raw_state.crane_last_operations)
        ):
            raise ValueError("Crane positions, load indicators, and operation times must align.")

        features = []
        for position, has_load, last_operation in zip(
            raw_state.crane_positions,
            raw_state.crane_loads,
            raw_state.crane_last_operations,
        ):
            features.extend(
                [
                    position[0] / 100.0,
                    position[1] / 50.0,
                    float(has_load),
                    last_operation / 200.0,
                    0.0,
                    0.5,
                ]
            )

        features.extend(
            [
                raw_state.vessel_import_containers / 50.0,
                raw_state.vessel_export_containers / 50.0,
                raw_state.vessel_stability_margin,
                raw_state.vessel_completion,
            ]
        )
        features.extend(float(value) for value in raw_state.hatch_states[:8])
        features.extend(
            [
                raw_state.truck_queue_length / 20.0,
                raw_state.yard_utilization,
                raw_state.weather_factor,
                raw_state.time_pressure,
                raw_state.current_dual_ratio,
                raw_state.estimated_makespan / 300.0,
                raw_state.priority_containers / 10.0,
                raw_state.interference_risk,
            ]
        )

        features = features[: self.state_dim]
        features.extend([0.0] * (self.state_dim - len(features)))
        return np.asarray(features, dtype=np.float32)

    def encode_expert_action(self, action: ExpertAction) -> np.ndarray:
        """Encode action attributes in a consistent eight-component vector."""
        if action.operation_type not in range(4):
            raise ValueError("operation_type must be one of 0, 1, 2, or 3.")

        return np.asarray(
            [
                action.crane_id / 10.0,
                action.operation_type,
                action.target_container / 100.0,
                action.priority_level,
                action.estimated_time / 60.0,
                float(action.enables_dual_cycle),
                action.stability_impact,
                action.interference_risk,
            ],
            dtype=np.float32,
        )

    @staticmethod
    def teacher_action(state: ExpertState, step: int, trajectory_id: int) -> ExpertAction:
        """Apply a transparent synthetic labeling rule, not an optimal solver."""
        operation_type = (
            0
            if state.vessel_import_containers > state.vessel_export_containers
            else 1
        )
        crane_id = int(np.argmin([position[0] for position in state.crane_positions]))
        enables_dual_cycle = (
            state.current_dual_ratio >= 0.75
            and state.interference_risk < 0.2
            and any(state.hatch_states)
        )
        return ExpertAction(
            crane_id=crane_id,
            operation_type=operation_type,
            target_container=(step * 7 + trajectory_id) % 100,
            priority_level=state.priority_containers / 10.0,
            estimated_time=3.0 + 12.0 * (1.0 - state.weather_factor),
            enables_dual_cycle=enables_dual_cycle,
            stability_impact=(0.5 - state.vessel_stability_margin) * 0.2,
            interference_risk=state.interference_risk,
        )

    def generate_expert_demonstrations(
        self, num_trajectories: int = 500
    ) -> List[ExpertTrajectory]:
        """Generate synthetic teacher demonstrations with the stated feature ranges."""
        if num_trajectories <= 0:
            raise ValueError("num_trajectories must be positive.")

        print(f"Generating {num_trajectories} synthetic teacher demonstrations...")
        trajectories = []
        for trajectory_id in range(num_trajectories):
            num_steps = np.random.randint(15, 45)
            import_containers = np.random.randint(20, 50)
            export_containers = np.random.randint(15, 40)
            states, actions, rewards = [], [], []

            for step in range(num_steps):
                state = ExpertState(
                    crane_positions=[
                        (np.random.uniform(0, 100), np.random.uniform(0, 50))
                        for _ in range(3)
                    ],
                    crane_loads=[bool(value) for value in (np.random.random(3) > 0.5)],
                    crane_last_operations=np.random.uniform(0, 200, size=3).tolist(),
                    vessel_import_containers=max(0, import_containers - step * 2),
                    vessel_export_containers=max(0, export_containers - step),
                    vessel_stability_margin=np.random.uniform(0.1, 1.0),
                    vessel_completion=step / num_steps,
                    hatch_states=[bool(value) for value in (np.random.random(8) > 0.3)],
                    truck_queue_length=int(np.random.randint(0, 20)),
                    yard_utilization=np.random.uniform(0.3, 0.9),
                    weather_factor=np.random.uniform(0.7, 1.0),
                    time_pressure=np.random.uniform(0.0, 1.0),
                    current_dual_ratio=np.random.uniform(0.6, 0.9),
                    estimated_makespan=np.random.uniform(100, 300),
                    priority_containers=int(np.random.randint(0, 10)),
                    interference_risk=np.random.uniform(0.0, 0.5),
                )
                action = self.teacher_action(state, step, trajectory_id)

                # This illustrative reward ranks rule-defined actions; it is not an IRL target.
                reward = 1.0
                reward += 2.0 if action.enables_dual_cycle else 0.0
                reward += 1.0 if action.operation_type in (0, 1) else 0.0
                reward += 0.5 if action.interference_risk < 0.1 else 0.0
                states.append(state)
                actions.append(action)
                rewards.append(reward)

            trajectories.append(
                ExpertTrajectory(states, actions, rewards, trajectory_id)
            )

        average_steps = np.mean([len(trajectory.states) for trajectory in trajectories])
        print(f"Generated {len(trajectories)} trajectories, averaging {average_steps:.1f} steps each.")
        return trajectories

    def train_behavioral_cloning(
        self, expert_trajectories: List[ExpertTrajectory], epochs: int = 100
    ) -> None:
        """Fit the policy to training demonstrations using minibatch SGD."""
        if not expert_trajectories or epochs <= 0:
            raise ValueError("Provide trajectories and a positive number of epochs.")

        state_rows, action_rows = [], []
        for trajectory in expert_trajectories:
            if len(trajectory.states) != len(trajectory.actions):
                raise ValueError("Each trajectory must have one action per state.")
            for state, action in zip(trajectory.states, trajectory.actions):
                state_rows.append(self.extract_state_features(state))
                action_rows.append(self.encode_expert_action(action))

        states = np.asarray(state_rows, dtype=np.float32)
        actions = np.asarray(action_rows, dtype=np.float32)
        if len(states) == 0:
            raise ValueError("The training set contains no state-action examples.")

        states = self.state_scaler.fit_transform(states)
        target_discrete = np.zeros((len(actions), 4), dtype=np.float32)
        operation_types = actions[:, 1].astype(int)
        target_discrete[np.arange(len(actions)), operation_types] = 1.0
        target_continuous = actions[:, 4:]

        self.training_history = {"bc_loss": [], "accuracy": []}
        batch_size = min(64, len(states))
        print(f"Training behavioral cloning for {epochs} epochs...")
        for epoch in range(epochs):
            indices = np.random.choice(len(states), batch_size, replace=False)
            batch_states = states[indices]
            batch_discrete = target_discrete[indices]
            batch_continuous = target_continuous[indices]

            loss = self.bc_network.backward(
                batch_states, batch_discrete, batch_continuous
            )
            probabilities, _ = self.bc_network.predict(batch_states)
            accuracy = np.mean(
                np.argmax(probabilities, axis=1)
                == np.argmax(batch_discrete, axis=1)
            )
            self.training_history["bc_loss"].append(loss)
            self.training_history["accuracy"].append(float(accuracy))

            if epoch % 20 == 0:
                print(f"Epoch {epoch:3d}: loss = {loss:.4f}, minibatch accuracy = {accuracy:.3f}")

        self.is_trained = True
        print(
            "Training complete. Final minibatch accuracy: "
            f"{self.training_history['accuracy'][-1]:.3f}"
        )

    def predict_action(self, state: ExpertState) -> Tuple[np.ndarray, np.ndarray]:
        """Return operation probabilities and four predicted action parameters."""
        if not self.is_trained:
            raise RuntimeError("Train the policy before requesting predictions.")

        features = self.extract_state_features(state).reshape(1, -1)
        normalized_state = self.state_scaler.transform(features)
        probabilities, continuous_parameters = self.bc_network.predict(normalized_state)
        return probabilities[0], continuous_parameters[0]

    def evaluate_performance(
        self, test_trajectories: List[ExpertTrajectory]
    ) -> Dict[str, float]:
        """Measure held-out label agreement and wall-clock prediction latency."""
        if not test_trajectories:
            raise ValueError("The test set must contain at least one trajectory.")

        correct_operations = 0
        correct_dual_flags = 0
        total_actions = 0
        decision_times = []

        for trajectory in test_trajectories:
            for state, teacher_action in zip(trajectory.states, trajectory.actions):
                start_time = time.perf_counter()
                probabilities, parameters = self.predict_action(state)
                decision_times.append((time.perf_counter() - start_time) * 1000.0)

                predicted_operation = int(np.argmax(probabilities))
                predicted_dual_flag = parameters[1] >= 0.5
                correct_operations += predicted_operation == teacher_action.operation_type
                correct_dual_flags += predicted_dual_flag == teacher_action.enables_dual_cycle
                total_actions += 1

        results = {
            "action_accuracy": correct_operations / total_actions,
            "dual_cycle_flag_accuracy": correct_dual_flags / total_actions,
            "avg_decision_time_ms": float(np.mean(decision_times)),
            "total_actions": total_actions,
        }
        print("Held-out evaluation against synthetic teacher labels:")
        print(f"  Operation accuracy: {results['action_accuracy']:.3f}")
        print(f"  Dual-cycle-flag accuracy: {results['dual_cycle_flag_accuracy']:.3f}")
        print(f"  Mean prediction latency: {results['avg_decision_time_ms']:.3f} ms")
        print("  These metrics measure label agreement, not operational efficiency.")
        return results


## Configure the learner

The network dimensions define the feature input, encoded action output, and hidden layers. The 42 input features include the state fields and padding described above. The eight outputs comprise four operation logits and four regression values. The architecture is kept explicit so that its role in the learning pipeline is easy to inspect.

In [ ]:
il_framework = DualCycleImitationLearner(
    state_dim=42,
    action_dim=8,
    hidden_dim=256,
)

print("=== Imitation Learning Configuration ===")
print(f"State dimension: {il_framework.state_dim}")
print(f"Action dimension: {il_framework.action_dim}")
print(f"Hidden layer sizes: {il_framework.hidden_dim}, {il_framework.hidden_dim // 2}")
print(
    "Network architecture: "
    f"{il_framework.state_dim} -> {il_framework.hidden_dim} -> "
    f"{il_framework.hidden_dim // 2} -> {il_framework.action_dim}"
)

## Generate and split demonstrations

The generator creates 500 synthetic trajectories using the feature ranges in its code. Its teacher rule prioritizes unloading when remaining imports exceed remaining exports, otherwise loading; it also assigns a dual-cycle flag using the current dual ratio, interference risk, and hatch state. These are explicit labeling rules, not demonstrations from port operators or an optimal scheduling algorithm.

The dataset is split by trajectory with a fixed split seed. Counting actual state-action pairs avoids treating an approximate number of steps per trajectory as observed data.

In [ ]:
expert_trajectories = il_framework.generate_expert_demonstrations(
    num_trajectories=500
)

total_steps = sum(len(trajectory.states) for trajectory in expert_trajectories)
dual_cycle_actions = sum(
    action.enables_dual_cycle
    for trajectory in expert_trajectories
    for action in trajectory.actions
)
dual_cycle_ratio = dual_cycle_actions / total_steps

train_trajectories, test_trajectories = train_test_split(
    expert_trajectories,
    test_size=0.2,
    random_state=42,
)

print("\n=== Synthetic Dataset Summary ===")
print(f"Trajectories: {len(expert_trajectories)}")
print(f"State-action steps: {total_steps}")
print(f"Teacher dual-cycle flags: {dual_cycle_actions}")
print(f"Teacher dual-cycle-flag share: {dual_cycle_ratio:.1%}")
print(f"Training trajectories: {len(train_trajectories)}")
print(f"Test trajectories: {len(test_trajectories)}")

## Train and evaluate behavioral cloning

Training minimizes the mathematical loss using minibatches of up to 64 state-action examples. The test set is not used to fit the feature scaler or update network weights. Held-out operation accuracy measures exact agreement with the synthetic teacher's operation labels. Dual-cycle-flag accuracy measures agreement for the flag predicted by the regression head after thresholding at 0.5. Neither score measures completed dual cycles, schedule quality, or safety.

In [ ]:
il_framework.train_behavioral_cloning(train_trajectories, epochs=87)
test_results = il_framework.evaluate_performance(test_trajectories)

## Compare with the transparent teacher rule

The rule-based comparison applies the same stated labeling rule directly to each test state. It provides a reference for whether the neural network reproduces that rule; it is not an independent operational baseline. Because the teacher labels are generated by this rule, agreement with it should be interpreted as imitation fidelity rather than evidence that the policy is better than an expert or improves terminal performance.

In [ ]:
def rule_based_baseline(test_trajectories: List[ExpertTrajectory]) -> Dict[str, float]:
    """Apply the synthetic teacher rule directly to held-out states."""
    total_actions = 0
    correct_operations = 0
    correct_dual_flags = 0

    for trajectory in test_trajectories:
        for state, teacher_action in zip(trajectory.states, trajectory.actions):
            predicted_action = il_framework.teacher_action(
                state,
                step=0,
                trajectory_id=0,
            )
            correct_operations += (
                predicted_action.operation_type == teacher_action.operation_type
            )
            correct_dual_flags += (
                predicted_action.enables_dual_cycle
                == teacher_action.enables_dual_cycle
            )
            total_actions += 1

    if total_actions == 0:
        raise ValueError("The test set contains no state-action examples.")

    return {
        "action_accuracy": correct_operations / total_actions,
        "dual_cycle_flag_accuracy": correct_dual_flags / total_actions,
        "total_actions": total_actions,
    }


baseline_results = rule_based_baseline(test_trajectories)

print("=== Held-Out Label Agreement ===")
print(
    "Behavioral cloning: "
    f"operation accuracy {test_results['action_accuracy']:.1%}, "
    f"dual-cycle-flag accuracy {test_results['dual_cycle_flag_accuracy']:.1%}"
)
print(
    "Teacher rule:        "
    f"operation accuracy {baseline_results['action_accuracy']:.1%}, "
    f"dual-cycle-flag accuracy {baseline_results['dual_cycle_flag_accuracy']:.1%}"
)
print("These figures compare prediction with synthetic labels, not crane-operation outcomes.")

## Visual diagnostics

The plots show minibatch training loss and accuracy, held-out agreement scores, measured prediction latency, and the training curves. The two agreement measures are label-prediction metrics, not physical efficiency measures. Prediction latency is measured on the current runtime and hardware; it is not a guarantee for real-time deployment. The teacher-rule panel is included as a reference, not as an independent competitor.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 11))

epochs = np.arange(1, len(il_framework.training_history["bc_loss"]) + 1)
axes[0, 0].plot(epochs, il_framework.training_history["bc_loss"], color="blue")
axes[0, 0].set_xlabel("Epoch")
axes[0, 0].set_ylabel("Behavioral cloning loss")
axes[0, 0].set_title("Training loss")
axes[0, 0].grid(True, alpha=0.3)

axes[0, 1].plot(epochs, il_framework.training_history["accuracy"], color="green")
axes[0, 1].set_xlabel("Epoch")
axes[0, 1].set_ylabel("Minibatch operation accuracy")
axes[0, 1].set_title("Training minibatch agreement")
axes[0, 1].set_ylim(0, 1)
axes[0, 1].grid(True, alpha=0.3)

score_names = ["Operation labels", "Dual-cycle flags"]
policy_scores = [
    test_results["action_accuracy"],
    test_results["dual_cycle_flag_accuracy"],
]
rule_scores = [
    baseline_results["action_accuracy"],
    baseline_results["dual_cycle_flag_accuracy"],
]
positions = np.arange(len(score_names))
width = 0.36
axes[0, 2].bar(positions - width / 2, policy_scores, width, label="Learned policy")
axes[0, 2].bar(positions + width / 2, rule_scores, width, label="Teacher rule")
axes[0, 2].set_xticks(positions, score_names)
axes[0, 2].set_ylabel("Held-out label accuracy")
axes[0, 2].set_title("Agreement with synthetic teacher")
axes[0, 2].set_ylim(0, 1)
axes[0, 2].legend()
axes[0, 2].grid(True, alpha=0.3, axis="y")

axes[1, 0].bar(
    ["Learned policy"],
    [test_results["avg_decision_time_ms"]],
    color="orange",
)
axes[1, 0].set_ylabel("Mean prediction latency (ms)")
axes[1, 0].set_title("Measured inference time")
axes[1, 0].grid(True, alpha=0.3, axis="y")

recent_count = min(50, len(epochs))
recent_epochs = epochs[-recent_count:]
left_axis = axes[1, 1]
right_axis = left_axis.twinx()
accuracy_line = left_axis.plot(
    recent_epochs,
    il_framework.training_history["accuracy"][-recent_count:],
    color="blue",
    label="Minibatch accuracy",
)
loss_line = right_axis.plot(
    recent_epochs,
    il_framework.training_history["bc_loss"][-recent_count:],
    color="red",
    label="Training loss",
)
left_axis.set_xlabel("Epoch")
left_axis.set_ylabel("Minibatch accuracy", color="blue")
right_axis.set_ylabel("Training loss", color="red")
left_axis.set_title("Recent training history")
left_axis.grid(True, alpha=0.3)
left_axis.legend(accuracy_line + loss_line, ["Minibatch accuracy", "Training loss"])

operation_names = ["Unload", "Load", "Move", "Wait"]
operation_counts = np.zeros(4, dtype=int)
for trajectory in train_trajectories:
    for action in trajectory.actions:
        operation_counts[action.operation_type] += 1
axes[1, 2].bar(operation_names, operation_counts, color="teal")
axes[1, 2].set_xlabel("Teacher operation type")
axes[1, 2].set_ylabel("Training examples")
axes[1, 2].set_title("Training label counts")
axes[1, 2].grid(True, alpha=0.3, axis="y")

fig.suptitle("Behavioral Cloning Diagnostics", fontsize=16)
fig.tight_layout()
plt.show()

## Results interpretation

The generated dataset and trained model produce the values displayed by the preceding cells when the notebook is run. No training output or numerical performance result is assumed here.

- **Training loss and minibatch accuracy** describe fit to sampled synthetic training examples. They are not independent estimates of real-world performance.
- **Held-out operation accuracy** measures whether the model predicts the teacher's operation class on test trajectories. **Dual-cycle-flag accuracy** measures agreement with the teacher's binary flag after thresholding its regression output. Neither metric demonstrates that a dual cycle is feasible or completed.
- **Teacher-rule comparison** checks imitation against the rule used to create labels. It is a consistency reference, not a comparison against an independent expert, solver, or operational baseline.
- **Prediction latency** is measured with the local runtime clock. It depends on the execution environment and includes only the policy prediction call, not data collection, safety validation, or crane control.
- **Generalization limits** are substantial: the test trajectories share the same synthetic generator and teacher rule as the training data. Evaluation on real vessel profiles would require representative data and a validated operational environment.

The implementation illustrates the behavioral-cloning portion of an imitation-learning workflow. An IRL experiment would need expert and comparison trajectories with a defined reward-learning procedure. A DAgger experiment would additionally need a simulator or environment that supplies policy-visited states and an expert oracle that labels them. Those ingredients are not provided by this synthetic example, so neither method is claimed as a measured result.

In [ ]:
print("=" * 68)
print("DUAL-CYCLING QUAY CRANE PROBLEM — TIER 4 SUMMARY")
print("=" * 68)
print("Method: Behavioral cloning of a synthetic teacher rule")
print(f"Dataset: {len(expert_trajectories)} trajectories, {total_steps} state-action steps")
print(
    "Network: "
    f"{il_framework.state_dim} -> {il_framework.hidden_dim} -> "
    f"{il_framework.hidden_dim // 2} -> {il_framework.action_dim}"
)
print(f"Training epochs: {len(il_framework.training_history['bc_loss'])}")
print("\nHELD-OUT LABEL AGREEMENT")
print(f"  Operation accuracy: {test_results['action_accuracy']:.1%}")
print(f"  Dual-cycle-flag accuracy: {test_results['dual_cycle_flag_accuracy']:.1%}")
print(f"  Mean prediction latency: {test_results['avg_decision_time_ms']:.3f} ms")
print(f"  Evaluated actions: {test_results['total_actions']}")
print("\nSCOPE")
print("  Synthetic labels are not real expert demonstrations or optimal schedules.")
print("  IRL and DAgger are not trained in this notebook.")
print("  Label accuracy does not establish safety or operational efficiency.")
print("=" * 68)